In [2]:
import os
import numpy as np
import torch

from cosmos_predict2.utils.printer import print_batch
from cosmos_predict2.utils.vis_helpers import save_action_as_image
from imaginaire.utils.io import save_image_or_video
from imaginaire.lazy_config import instantiate

from cosmos_predict2.configs.expert.defaults.data_tcl import tcl_train_dataset, DataLoader

In [3]:
train_dataset = instantiate(
    tcl_train_dataset,
    p_camera_drop=0.,
    future_frame_skip=8,
)
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    drop_last=False
)

[TCLDataset] loaded key=rel_actions shape=(16247, 7) from /home/geyuan/datasets/TCL/1024_sweep_bean/extracted/rel_actions.npy
[TCLDataset] total length: 16247
[TCLDatasetHDF5] using h5 data_manager: /home/geyuan/datasets/TCL/hdf5/1024_sweep_bean_240p.h5 . total_len: 16247
[TCLDataset] loading dataset statistics from: /home/geyuan/datasets/TCL/1024_sweep_bean/statistics.json
[DEBUG] Loading language embeddings from: /home/geyuan/datasets/TCL/lang_emb_t5xxl/all/t5_embeddings.npz
[TCLImageDataset] dataset loaded, split=train, val_ratio=0.0, len=16247; meta_total_len=16247, norm_type=identity.
[TCLDataset] loaded key=rel_actions shape=(34523, 7) from /home/geyuan/datasets/TCL/1024_eggs_pick_place/extracted/rel_actions.npy
[TCLDataset] total length: 34523
[TCLDatasetHDF5] using h5 data_manager: /home/geyuan/datasets/TCL/hdf5/1024_eggs_pick_place_240p.h5 . total_len: 34523
[TCLDataset] loading dataset statistics from: /home/geyuan/datasets/TCL/1024_eggs_pick_place/statistics.json
[DEBUG] Loa

In [5]:
from tqdm import tqdm

vis_idx = 3
in_sample = None

for idx, batch in enumerate(tqdm(train_dataloader)):
    if idx == 0:
        print_batch('TCL', batch)
    if idx < vis_idx:
        continue
    # print_batch('TCL', batch)
    # print("language:", batch["task"]["language_instruction"])

    in_sample = batch
    break

  0%|                                                          | 1/11645 [00:00<1:29:55,  2.16it/s]

TCL: Dict, keys=['action', 'video', 'agent_pos', 'force', 'annotation_file', '__key__', 'lang_text', 't5_text_embeddings', 't5_text_mask', 'fps', 'image_size', 'num_frames', 'padding_mask', 'sample_n_views', 'view_indices', 'latent_view_indices_B_T', 'idx']
--action, <class 'torch.Tensor'>, shape=torch.Size([8, 32, 7]), min=-1.0000, max=1.0000
--video, <class 'torch.Tensor'>, shape=torch.Size([8, 3, 10, 160, 240]), min=0.0000, max=255.0000
--agent_pos, <class 'torch.Tensor'>, shape=torch.Size([8, 5, 6]), min=-6.3848, max=1.9174
--force, <class 'torch.Tensor'>, shape=torch.Size([8, 5, 6]), min=-1.0000, max=1.0000
--annotation_file: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=4, value='None'
--__key__: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=4, value='None'
--lang_text: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=76, value='Pick up the broom and sweep the coffee beans on the table into the dustpan.#'
--t5_text_embeddings,

  0%|                                                            | 3/11645 [00:00<47:45,  4.06it/s]


In [6]:
""" Visualization Remapped Dataloader """
max_vis_len = 50
mv_sample = in_sample
DEBUG_DATASET = "tcl"

horizon = mv_sample['agent_pos'][0].shape[0]
save_image_or_video(
    mv_sample['video'][0, :, :horizon].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentview.mp4",
    fps=5
)
save_image_or_video(
    mv_sample['video'][0, :, horizon:].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_gripper.mp4",
    fps=5
)

dataset_stats = train_dataset.dataset_stats

save_action_as_image(
    train_dataset.denorm_action(
        mv_sample['action'].cpu().numpy(), train_dataset.norm_action_type, dataset_stats['rel_actions']
    )[0, :, :7],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action0123456.mp4",
)
# save_action_as_image(
#     train_dataset.denorm_action(
#         mv_sample['action'], train_dataset.norm_action_type, dataset_stats['rel_actions']
#     )[0, :, 6:7],
#     f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action6.png",
# )

save_action_as_image(
    train_dataset.denorm_state_or_force(
        mv_sample['agent_pos'].cpu().numpy(), "mean", dataset_stats['robot_obs']
    )[0, :, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos012.png",
)
# save_action_as_image(
#     mv_sample['agent_pos'][:, -1:] * dataset_multi_view.meta_gripper_states_std[-1] + dataset_multi_view.meta_gripper_states_mean[-1],
#     "/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos6.png",
# )

''' Force '''
save_action_as_image(
    train_dataset.denorm_state_or_force(
        mv_sample['force'].numpy(), train_dataset.norm_force_type, dataset_stats['force_torque']
    )[0, :, :6],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_force012345.mp4",
)


Plotting action dynamic figures...
[DEBUG] save_3d_action_as_image: (5, 3) float64 -0.14906242489814758 0.5756394267082214
[11-04 16:22:08|INFO|../../../../../../../home/geyuan/code/cospred2nvidia/cosmos_predict2/utils/vis_helpers.py:197:save_3d_action_as_image] Saved 3D action trajectory visualization to /home/geyuan/code/cospred2nvidia/output/de_tcl_mv_agentpos012.png
Plotting action dynamic figures...
